# Введение в инструментарий анализа данных

## 1. Назначение практической работы

Цель работы - освоить полный базовый цикл анализа табличных данных: подготовить воспроизводимое Python-окружение, загрузить данные, проверить их структуру и качество, выполнить предобработку, получить описательные статистики, сформировать простые аналитические срезы, визуализировать закономерности и сохранить подготовленный набор данных в обменных и аналитических форматах.

В отличие от работы, построенной вокруг отдельных конструкций языка Python, здесь синтаксис используется только как инструмент решения аналитической задачи. Каждая операция должна сопровождаться интерпретацией результата.

Студент отвечает не только на вопрос «что вернула функция?», но и на вопрос «что это означает для исследуемого набора данных?».

In [ ]:
print("Hello, World!")


## 2. Рабочее окружение Python

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)


## 3. Минимальное введение в Python, NumPy и Pandas для анализа

### Векторное преобразование

In [ ]:
import numpy as np


wind_ms = np.array([1.14, 0.80, 0.97])
wind_kmh = wind_ms * 3.6

print(np.round(wind_kmh, 2))


### Ключевые объекты Pandas

In [ ]:
s = pd.Series([10, 20, 30], name="value")
frame = pd.DataFrame({
    "type": ["A", "B", "A"],
    "value": [10, 20, 30]
})

print(type(s).__name__)
print(type(frame).__name__)
print(frame.groupby("type")["value"].sum())


## 4. Форматы файлового хранения данных и загрузка CSV

In [ ]:
from pathlib import Path
import pandas as pd


DATA_PATH = Path("FiresRu.csv")
df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns.tolist())
df.head()


## 5. Первичная диагностика данных

### Структура и память

In [ ]:
import io


buf = io.StringIO()
df.info(buf=buf, memory_usage="deep")
print(buf.getvalue())


### Пропуски, дубликаты и уникальные значения

In [ ]:
print("Пропуски по столбцам:")
print(df.isna().sum())

print("Полных дубликатов:", df.duplicated().sum())

print("Число уникальных значений:")
print(df.nunique())


### Проверка соответствия кода и категории

In [ ]:
type_map = (
    df[["type_id", "type_name"]]
        .drop_duplicates()
        .sort_values("type_id")
)

print(type_map.to_string(index=False))


### Описательная статистика

In [ ]:
cols = [
    "lon", "lat", "temperature_c", "precipitation_mm",
    "relative_humidity", "wind_speed_ms", "solar_radiation"
]

print(df[cols].describe().round(2).T)


## 6. Типы данных и предобработка

### Оптимизация типов

In [ ]:
before = df.memory_usage(deep=True).sum()

clean_df = df.copy()
clean_df["type_name"] = clean_df["type_name"].astype("category")
clean_df["type_id"] = clean_df["type_id"].astype("int8")

after = clean_df.memory_usage(deep=True).sum()

print(f"До: {before / 1024**2:.2f} MiB")
print(f"После: {after / 1024**2:.2f} MiB")
print(f"Снижение: {(1 - after / before) * 100:.1f}%")


### Создание вычисляемых признаков

In [ ]:
clean_df["wind_speed_kmh"] = (
    clean_df["wind_speed_ms"] * 3.6
).round(2)

clean_df["is_forest_fire"] = (
    clean_df["type_name"] == "Forest fire"
)

print(clean_df[
    ["type_name", "wind_speed_ms",
     "wind_speed_kmh", "is_forest_fire"]
].head())


## 7. Фильтрация, группировка и агрегирование

### Распределение наблюдений по типам

In [ ]:
type_stats = (
    df["type_name"]
        .value_counts()
        .to_frame("n")
        .assign(share_pct=lambda x: x["n"] / len(df) * 100)
)

print(type_stats.round(2))


### Групповые средние

In [ ]:
group_stats = (
    df.groupby("type_name", observed=True)
        .agg(
            n=("type_name", "size"),
            temp_mean=("temperature_c", "mean"),
            precip_mean=("precipitation_mm", "mean"),
            humid_mean=("relative_humidity", "mean"),
            wind_mean=("wind_speed_ms", "mean"),
            solar_mean=("solar_radiation", "mean")
        )
        .sort_values("n", ascending=False)
)

print(group_stats.round(2))


### Условная фильтрация

In [ ]:
subset = df.loc[
    (df["relative_humidity"] < 40)
    & (df["wind_speed_ms"] > 2),
    ["type_name", "lon", "lat", "temperature_c",
    "relative_humidity", "wind_speed_ms"]
]

print("Строк в срезе:", len(subset))
print(subset.head().round(2))


## 8. Корреляция и потенциальные выбросы

### Корреляционная матрица

In [ ]:
weather = [
    "temperature_c", "precipitation_mm", "relative_humidity",
    "wind_speed_ms", "solar_radiation"
]

corr = df[weather].corr()
print(corr.round(3))


### Правило IQR на примере скорости ветра

In [ ]:
q1 = df["wind_speed_ms"].quantile(0.25)
q3 = df["wind_speed_ms"].quantile(0.75)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask = (
    (df["wind_speed_ms"] < lower) |
    (df["wind_speed_ms"] > upper)
)

mask_sum = mask.sum()
percentage = mask_sum / len(df) * 100

print(f"Q1={q1:.3f}; Q3={q3:.3f}; IQR={iqr:.3f}")
print(f"Границы: [{lower:.3f}; {upper:.3f}]")
print(f"Кандидатов: {mask_sum} ({percentage:.2f}% строк)")


## 9. Визуальный разведочный анализ

### Bar chart частоты типов событий

In [ ]:
import matplotlib.pyplot as plt


counts = df["type_name"].value_counts().sort_values()
ax = counts.plot(kind="barh", figsize=(8, 4.5))

ax.set_title("Количество наблюдений по типу события")
ax.set_xlabel("Число наблюдений")
ax.set_ylabel("Тип события")

plt.grid(True, axis="x", linestyle="-", alpha=0.7)
plt.tight_layout()
plt.show()


### Histogram: распределение температуры

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(df["temperature_c"], bins=30, edgecolor="white")

plt.title("Распределение температуры")
plt.xlabel("Температура, °C")
plt.ylabel("Частота")

plt.grid(True, axis="y", linestyle="-", alpha=0.7)
plt.tight_layout()
plt.show()


### Scatter plot для температуры и влажности

In [ ]:
sample = df.sample(n=5000, random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(
    sample["temperature_c"],
    sample["relative_humidity"],
    s=10,
    alpha=0.35
)

plt.title("Температура и относительная влажность")
plt.xlabel("Температура, °C")
plt.ylabel("Относительная влажность")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Пространственный scatter plot

In [ ]:
plt.figure(figsize=(8, 5.5))

for name, part in df.groupby("type_name"):
    plt.scatter(
        part["lon"], part["lat"],
        s=6, alpha=0.35, label=name
    )

plt.xlabel("Долгота")
plt.ylabel("Широта")
plt.title("Пространственное распределение наблюдений")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Экспорт CSV, JSON и Parquet

### Сохранение подготовленного DataFrame

In [ ]:
from pathlib import Path


OUT = Path("output")
OUT.mkdir(exist_ok=True)

clean_df.to_csv(
    OUT / "FiresRu_clean.csv",
    index=False,
    encoding="utf-8"
)

clean_df.to_json(
    OUT / "FiresRu_clean.json",
    orient="records",
    lines=True,
    force_ascii=False
)

clean_df.to_parquet(
    OUT / "FiresRu_clean.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy"
)


### Измерение размеров

In [ ]:
files = [
    OUT / "FiresRu_clean.csv",
    OUT / "FiresRu_clean.json",
    OUT / "FiresRu_clean.parquet"
]

for path in files:
    size_mib = path.stat().st_size / 1024**2
    print(f"{path.name:24s} {size_mib:7.2f} MiB")


### Проверка round-trip

In [ ]:
csv_back = pd.read_csv(OUT / "FiresRu_clean.csv")
parquet_back = pd.read_parquet(OUT / "FiresRu_clean.parquet")

print(csv_back.shape)
print(parquet_back.shape)
print(parquet_back.dtypes)
